# Pla2g2 — KNN alignment diagnostics

EmmaEmb metrics assess whether task-relevant information is readily accessible in the geometry
of the embedding space, as reflected in neighbourhood structure and pairwise distances. A high
alignment score suggests that simple downstream models are likely to exploit this structure
effectively.

This notebook applies a 5-step protocol to the Pla2g2 secreted phospholipase A2 classification
task, comparing **ESMC** and **ProtT5** embeddings on 446 sequences annotated with enzyme class,
gene family, and species.

| Step | Purpose |
|---|---|
| **1 — Geometry** | Anisotropy + hubness: characterise the space before choosing parameters |
| **2 — Labels** | Class distribution, n_min, imbalance ratio, per-class random baseline |
| **3 — k range** | Derive a justified k range from n_min, N, and hubness |
| **4 — Main analysis** | KNN alignment across k and distance metrics; ranking stability |
| **5 — Robustness** | Rebalancing and label-noise sweeps |
| **6 — Report** | Model ranking with confidence level and full parameter context |


In [3]:
import sys
from pathlib import Path
from itertools import combinations

sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from emmaemb.core import Emma
from emmaemb.functions import (
    get_knn_alignment_scores,
    get_anisotropy_diagnostics,
    get_hubness_diagnostics,
)
from emmaemb.vizualisation import (
    plot_knn_alignment_across_k,
    plot_knn_alignment_vs_class_balance,
    plot_knn_alignment_vs_feature_noise,
)
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from dim_reduction_utils import EMB_SPACES, EMBEDDINGS_PATH, load_row
from constants import SCPDB_DATASET, CRYPTOBENCH_TRAIN_DATASET

In [ ]:
import csv

feature_data = []
embeddings = {}

protein_num = 50
for dataset in [SCPDB_DATASET, CRYPTOBENCH_TRAIN_DATASET]:
    BINDING_TYPE = 'REGULAR' if dataset == SCPDB_DATASET else 'CRYPTIC'
    with open(dataset, 'r') as f:
        reader = csv.reader(f, delimiter=';')
        for i, row in enumerate(reader):
            if i > protein_num:
                break
            protein_id, annotation, sequence, these_embeddings = load_row(row)

            if protein_id is None:
                continue
            assert len(these_embeddings[EMB_SPACES[0][0]]) == len(sequence), f"Length of embeddings for {protein_id} does not match sequence length"
            for i in range(len(sequence)):
                if i in annotation:
                    feature_data.append([sequence[i], f'{BINDING_TYPE}-BINDING'])
                else:
                    feature_data.append([sequence[i], f'NON-BINDING'])

            for embeddings_name in these_embeddings:
                if embeddings_name not in embeddings:
                    embeddings[embeddings_name] = []
                embeddings[embeddings_name].append(these_embeddings[embeddings_name])

for embeddings_name in embeddings:
    concatenated_embeddings_path = f"{EMBEDDINGS_PATH}/concatenated-embeddings/{embeddings_name}_binding_site_embeddings.npy"
    embeddings[embeddings_name] = np.concatenate(embeddings[embeddings_name], axis=0)
    np.save(concatenated_embeddings_path, embeddings[embeddings_name])  

feature_data = pd.DataFrame.from_records(feature_data, columns=["amino acid", "binding_site"])

# initiate Emma object and load embedding spaces
ema = Emma(feature_data=feature_data)
for embeddings_name, _ in EMB_SPACES:
    ema.add_emb_space(
        embeddings_source=f"{EMBEDDINGS_PATH}/concatenated-embeddings/{embeddings_name}_binding_site_embeddings.npy",
        emb_space_name=embeddings_name)

In [6]:
models = ['ESM2', 'ProtT5', 'ProstT5', 'ANKH']
colors = {
    "ESM2":   "#8B0000",
    "ProtT5": "#303496",
    "ProstT5": "#008000",
    "ANKH": "#FFD700"
}
distance_metrics = ['euclidean', 'cosine']

In [61]:
# Compute pairwise distances
import pickle
with open('/media/drive2/vkrhk/embeddings/emma-for-emb-diagnostics.pkl', 'rb') as f:
    ema = pickle.load(f)
print(len(ema.metadata))  
# for metric in distance_metrics:
#     for model in models:
#         ema.calculate_pairwise_distances(emb_space=model, metric=metric)
#         print(f"  {model} / {metric} done")
# 
# import pickle
# with open('/media/drive2/vkrhk/embeddings/emma-for-emb-diagnostics.pkl', 'wb') as f:
#     pickle.dump(ema, f)

28884


---
## Step 1 — Characterise embedding space geometry

Run **before** choosing any parameters. Outputs: recommended distance metric and a hubness flag
that constrains the valid k range in Step 3.

### 1a — Anisotropy

In an isotropic embedding space, vectors point in many different directions and are spread roughly uniformly. In an anisotropic space, most vectors cluster in a narrow cone, i.e. they all point in roughly the same direction. This can be a known artifact in high-dimensional spaces (see Ethayarajh, 2019).

**Why it matters for EmmaEmb**: if all embeddings point in a similar direction, cosine similarities between all pairs will be artificially high and uniform, reducing the discriminative power of distance metrics. Euclidean and Manhattan distances are also affected because high-variance dimensions dominate. In anisotropic spaces, cosine distance is generally more robust than Euclidean because it normalises for magnitude, but if anisotropy is severe it too loses discriminative power.

We provide two complementary diagnostic tests to flag embedding spaces that may not meet the geometric requirements for reliable KNN-based analysis. If an embedding space passes both tests, this gives positive evidence that the key assumptions are likely satisfied, though it is not a guarantee that all geometric properties are met. The hubness diagnostics in section 1.2 provide a further independent check, and both should be considered together before proceeding.

**Test 1 — Average pairwise cosine similarity**: mean cosine similarity between random embedding pairs. Should be close to 0 in an isotropic space. High values indicate a global directional bias.

**Test 2 — Per-dimension variance**: checks whether a small number of dimensions account for most of the variance. Can occur even when Test 1 passes, and specifically affects Euclidean and Manhattan distances.

Test 1 is run on both raw and mean-centred embeddings. **Mean centering** removes the global mean direction. If the anisotropy score improves substantially after centering, the anisotropy is a mean-offset effect we will in this notebook automatically apply mean centering to the affected embedding space and uses the centred embeddings for all subsequent steps in the workflow, including the hubness diagnostics. If the score remains high after centering, the anisotropy is likely structural, the raw embeddings are carried forward and this is flagged in the summary output.


In [62]:
from plotly.offline import init_notebook_mode, iplot
from plotly.graph_objs import *

init_notebook_mode(connected=True)         # initiate notebook for offline plot

In [63]:
aniso = get_anisotropy_diagnostics(ema, n_pairs=10_000, seed=42)
print(aniso)


=== Anisotropy Diagnostics ===

Raw average cosine similarity between random pairs. Near 0 = isotropic. Above 0
= embeddings cluster in a narrow cone. Other metrics: AnisotropyScore
(dimensionality-corrected, comparable across models), ParticipationRatio
(effective number of dimensions in use), MeanDimVariance ± StdDimVariance
(spread and unevenness across dimensions). Full table available in
.data['summary'].

Summary:
  ESM2: cosine sim = 0.474  |  mean-centred = 0.005  (resolves anisotropy)  —  dim = 1280, participation ratio = 81.8, mean var = 0.0397 (±0.1520)
  ANKH: cosine sim = 0.029  |  mean-centred = 0.001  (resolves anisotropy)  —  dim = 1536, participation ratio = 673.2, mean var = 0.0004 (±0.0005)
  ProstT5: cosine sim = 0.036  |  mean-centred = 0.001  (resolves anisotropy)  —  dim = 1024, participation ratio = 643.7, mean var = 0.0453 (±0.0348)
  ProtT5: cosine sim = 0.016  —  dim = 1024, participation ratio = 981.5, mean var = 0.0368 (±0.0077)


In [64]:
df_aniso = aniso.data["summary"]

# Build long-form df: original + MC where available
df_orig = df_aniso[["Embedding", "AvgPairwiseCosineSim"]].copy()
df_orig["Type"] = "Original"
df_orig = df_orig.rename(columns={"AvgPairwiseCosineSim": "Score"})

df_mc = df_aniso[["Embedding", "AvgPairwiseCosineSim_MC"]].dropna(subset=["AvgPairwiseCosineSim_MC"]).copy()
df_mc["Type"] = "Mean-centred"
df_mc = df_mc.rename(columns={"AvgPairwiseCosineSim_MC": "Score"})

df_plot = pd.concat([df_orig, df_mc], ignore_index=True)
type_colors = {"Original": "#303496", "Mean-centred": "#B4063D"}

fig = px.bar(
    df_plot, x="Embedding", y="Score", color="Type", barmode="group",
    text=df_plot["Score"].map("{:.3f}".format),
    title="Anisotropy — average pairwise cosine similarity",
    labels={"Score": "Avg cosine similarity", "Embedding": "", "Type": ""},
    template="plotly_white",
    color_discrete_map=type_colors,
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis=dict(range=[0, df_plot["Score"].max() * 1.2]),
    font=dict(family="Arial", size=14),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.add_hline(y=0, line_dash="dot", line_color="grey",
              annotation_text="isotropic baseline (0)", annotation_position="bottom right")
fig.update_xaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.update_yaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.show()


In [65]:
var_dict = aniso.data["variance_per_dim"]

fig = go.Figure()
for model, var in var_dict.items():
    sorted_var = np.sort(var)[::-1]
    cumvar = np.cumsum(sorted_var) / sorted_var.sum()
    n90 = int(np.searchsorted(cumvar, 0.90)) + 1
    print(f"{model}: {n90} dims capture 90% of variance (out of {len(var)})")
    fig.add_trace(go.Scatter(
        x=np.arange(len(cumvar)), y=cumvar,
        mode="lines", name=model,
        line=dict(color=colors[model], width=2),
    ))

fig.add_hline(y=0.90, line_dash="dash", line_color="grey",
              annotation_text="90%", annotation_position="right")
fig.update_layout(
    title="Cumulative explained variance",
    xaxis_title="Number of dimensions",
    yaxis_title="Cumulative variance fraction",
    yaxis=dict(range=[0, 1.05]),
    template="plotly_white",
    font=dict(family="Arial", size=14),
)
fig.update_xaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.update_yaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.show()


ESM2: 1089 dims capture 90% of variance (out of 1280)
ANKH: 1247 dims capture 90% of variance (out of 1536)
ProstT5: 827 dims capture 90% of variance (out of 1024)
ProtT5: 868 dims capture 90% of variance (out of 1024)


---
### Mean-centering

Anisotropy was detected above threshold in at least one embedding space.
Mean-centering subtracts the per-dimension mean, shifting the cloud to the origin.
All cached pairwise distances and projections are cleared automatically.


In [66]:
# Apply mean-centering to all embedding spaces flagged as anisotropic
anisotropic_spaces = (
    aniso.data["summary"]
    .dropna(subset=["AnisotropyScore_MC"])["Embedding"]
    .tolist()
)
print(f"Applying mean-centering to: {anisotropic_spaces}")
# ema.mean_center(emb_spaces=anisotropic_spaces)
ema.mean_center(emb_spaces=['ESM2'])

# Recompute pairwise distances on the centred embeddings
for metric in distance_metrics:
    for model in anisotropic_spaces:
        ema.calculate_pairwise_distances(emb_space=model, metric=metric)
        print(f"  {model} / {metric} done")


Applying mean-centering to: ['ESM2', 'ANKH', 'ProstT5']
'ESM2' mean-centred. Cached distances and projections cleared.
Calculating pairwise distances using euclidean...
  ESM2 / euclidean done
Pairwise distances using euclidean already calculated.
  ANKH / euclidean done
Pairwise distances using euclidean already calculated.
  ProstT5 / euclidean done
Calculating pairwise distances using cosine...
  ESM2 / cosine done
Pairwise distances using cosine already calculated.
  ANKH / cosine done
Pairwise distances using cosine already calculated.
  ProstT5 / cosine done


### 1b — Hubness

Hubness is a phenomenon in high-dimensional spaces where a small number of points become the nearest neighbours of a disproportionately large number of other points, not because they are genuinely similar to everything, but as a mathematical consequence of dimensionality. It is often worsened by anisotropy (Radovanović et al., 2010).

**Why it matters for EmmaEmb**: if a few hub points dominate all neighbourhoods, KNN alignment scores reflect hub structure rather than genuine representational differences. Model rankings based on these scores may therefore be unreliable.

We assess hubness using the k-occurrence distribution, counting how many times each point appears in the k-nearest neighbour sets of all other points. In a well-behaved space this distribution is roughly uniform. In a high-dimensional or anisotropic space it becomes heavily right-skewed.

Two summary statistics are reported:
- **Skewness of the k-occurrence distribution**: captures the shape. High positive skewness indicates a long right tail of hub points.
- **Robin Hood index**: measures the inequality of the distribution: the fraction of total neighbour occurrences that would need to be redistributed from hubs to underrepresented points to equalise it. Originally from income inequality measurement but already established in the hubness context (Feldbauer et al., 2018). A value of 0 means perfect equality; high values mean a few hubs dominate.


In [67]:
k_hub = 10
hub = get_hubness_diagnostics(ema, k=k_hub, metric="cosine")
print(hub)


=== Hubness Diagnostics ===

Hubness is the tendency for a small number of points ('hubs') to appear
disproportionately often as k-nearest neighbours of other points — a geometric
phenomenon that grows with dimensionality. The k-occurrence distribution shows
how many times each point was selected as a neighbour; a heavy right tail
indicates hub points. The Robin Hood index (RHI) quantifies this inequality: 0 =
perfectly uniform occurrence, ~1 = extreme hubness.

Summary:
  ESM2: Robin Hood index = 0.337
  ANKH: Robin Hood index = 0.232
  ProstT5: Robin Hood index = 0.208
  ProtT5: Robin Hood index = 0.196


In [68]:
rhi_df = hub.data["rhi"]

fig = px.bar(
    rhi_df, x="Embedding", y="RobinHoodIndex",
    color="Embedding",
    text=rhi_df["RobinHoodIndex"].map("{:.3f}".format),
    title=f"Robin Hood index (k={k_hub}, cosine)",
    labels={"RobinHoodIndex": "Robin Hood index", "Embedding": ""},
    template="plotly_white",
    color_discrete_map=colors,
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis=dict(range=[0, rhi_df["RobinHoodIndex"].max() * 1.35]),
    showlegend=False, font=dict(family="Arial", size=14),
)
fig.update_xaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.update_yaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.show()


In [69]:
kocc_df = hub.data["k_occurrence"]

fig = px.histogram(
    kocc_df, x="KOccurrence", color="Embedding",
    barmode="overlay", nbins=20, opacity=0.65,
    title=f"k-occurrence distribution (k={k_hub}, cosine)",
    labels={"KOccurrence": "k-occurrence count"},
    template="plotly_white",
    color_discrete_map=colors,
)
fig.add_vline(x=k_hub, line_dash="dot", line_color="grey",
              annotation_text=f"expected mean = {k_hub} (i.e. k)",
              annotation_position="top right")
fig.update_layout(font=dict(family="Arial", size=14))
fig.update_xaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.update_yaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.show()


### 1c — Cross-space pairwise distance correlation

Scatter of cosine distances for all unique sequence pairs across embedding spaces.
Agreement between spaces (high Spearman ρ) means the models impose similar geometry on
the data; divergence points to model-specific structure worth investigating.

In [70]:
# tohle bezi hodinu lol, skip


# model_pairs = list(combinations(models, 2))
# n_pairs_plot = len(model_pairs)
# n_samples = len(ema.sample_names)
# idx_i, idx_j = np.triu_indices(n_samples, k=1)
# 
# fig = make_subplots(
#     rows=1, cols=n_pairs_plot,
#     subplot_titles=[f"{a} vs {b}" for a, b in model_pairs],
# )
# 
# for col_idx, (model_a, model_b) in enumerate(model_pairs, start=1):
#     d_a = ema.emb[model_a]["pairwise_distances"]["cosine"][idx_i, idx_j]
#     d_b = ema.emb[model_b]["pairwise_distances"]["cosine"][idx_i, idx_j]
# 
#     rho, p_val = stats.spearmanr(d_a, d_b)
#     p_text = f"{p_val:.2e}" if p_val < 0.001 else f"{p_val:.4f}"
# 
#     ax_max = float(np.ceil(max(d_a.max(), d_b.max()) * 10) / 10)  # round up to 1 d.p.
# 
#     fig.add_trace(
#         go.Scattergl(
#             x=d_a, y=d_b,
#             mode="markers",
#             marker=dict(size=3, color="#AAAAAA", opacity=0.3),
#             showlegend=False,
#             hoverinfo="skip",
#         ),
#         row=1, col=col_idx,
#     )
# 
#     # Spearman annotation in top-left of each subplot
#     xref = "x domain" if col_idx == 1 else f"x{col_idx} domain"
#     yref = "y domain" if col_idx == 1 else f"y{col_idx} domain"
#     fig.add_annotation(
#         xref=xref, yref=yref,
#         x=0.04, y=0.97,
#         text=f"ρ = {rho:.2f}<br>p = {p_text}",
#         showarrow=False,
#         font=dict(size=13),
#         align="left",
#         bgcolor="rgba(255,255,255,0.75)",
#     )
# 
#     fig.update_xaxes(
#         title_text=f"Cosine distance<br>{model_a}",
#         range=[0, ax_max], showgrid=False,
#         showline=True, linecolor="black", linewidth=1.5,
#         row=1, col=col_idx,
#     )
#     fig.update_yaxes(
#         title_text=f"Cosine distance<br>{model_b}",
#         range=[0, ax_max], showgrid=False,
#         showline=True, linecolor="black", linewidth=1.5,
#         row=1, col=col_idx,
#     )
# 
# cell_size = 420
# fig.update_layout(
#     template="plotly_white",
#     font=dict(family="Arial", size=14),
#     width=cell_size * n_pairs_plot,
#     height=cell_size,
#     margin=dict(l=10, r=10, t=40, b=10),
# )
# fig.show()

In [71]:
# ── Step 1 output ─────────────────────────────────────────────────────────────
df_aniso_s = aniso.data["summary"]
rhi_s = hub.data["rhi"].set_index("Embedding")

print("Recommended distance metric:")
for _, row in df_aniso_s.iterrows():
    cos = row["AvgPairwiseCosineSim"]
    mc  = row.get("MeanCenteringHelps", False)
    if cos > 0.3:
        rec = "cosine  ⚠  high anisotropy"
    elif cos > 0.1:
        rec = "cosine  (mild anisotropy)"
    else:
        rec = "cosine / euclidean / manhattan (isotropic)"
    mc_note = "  → mean-centering applied" if mc else ""
    print(f"  {row['Embedding']:12s}  avg cosine sim = {cos:.3f}  → {rec}{mc_note}")

print()
print("Hubness flags:")
_hubness_high = False
for emb, row in rhi_s.iterrows():
    rhi = row["RobinHoodIndex"]
    if rhi > 0.3:
        flag = "⚠ high — use smaller k"
        _hubness_high = True
    else:
        flag = "OK"
    print(f"  {emb:12s}  RHI = {rhi:.3f}  → {flag}")


Recommended distance metric:
  ESM2          avg cosine sim = 0.474  → cosine  ⚠  high anisotropy  → mean-centering applied
  ANKH          avg cosine sim = 0.029  → cosine / euclidean / manhattan (isotropic)  → mean-centering applied
  ProstT5       avg cosine sim = 0.036  → cosine / euclidean / manhattan (isotropic)  → mean-centering applied
  ProtT5        avg cosine sim = 0.016  → cosine / euclidean / manhattan (isotropic)

Hubness flags:
  ESM2          RHI = 0.337  → ⚠ high — use smaller k
  ANKH          RHI = 0.232  → OK
  ProstT5       RHI = 0.208  → OK
  ProtT5        RHI = 0.196  → OK


---
## Step 2 — Characterise the label distribution

Count samples per class, identify **n_min** (hard upper bound on k), compute the imbalance ratio,
and derive per-class random baselines. All alignment scores must be interpreted relative to these
baselines.


In [72]:
metadata = feature_data
feature = "binding_site"
metadata = ema.metadata

class_counts = metadata[feature].value_counts().reset_index()
class_counts.columns = [feature, "count"]

fig = px.bar(
    class_counts, x=feature, y="count",
    text="count",
    title=f"Class distribution — Pla2g2 ({feature})",
    template="plotly_white",
    color=feature,
)
fig.update_traces(textposition="outside")
fig.update_layout(font=dict(family="Arial", size=13), showlegend=False)
fig.update_xaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.update_yaxes(showline=True, linecolor="black", linewidth=2, showgrid=False)
fig.show()

# ── Step 2 output ──────────────────────────────────────────────────────────────
N_total   = int(class_counts["count"].sum())
n_min     = int(class_counts["count"].min())
n_max     = int(class_counts["count"].max())
imbalance = n_max / n_min
n_classes = int(class_counts[feature].nunique())

print(f"N = {N_total}, classes = {n_classes}, n_min = {n_min}, n_max = {n_max}")
print(f"Imbalance ratio: {imbalance:.1f}x  ({'⚠ flag — run robustness sweep in Step 5' if imbalance > 3 else '✓ moderate'})")
print(f"Uniform random baseline: 1/{n_classes} = {1/n_classes:.3f}")
print()
print("Per-class random baseline  (n_class / (N−1)):")
for _, row in class_counts.iterrows():
    b = row["count"] / (N_total - 1)
    print(f"  {str(row[feature]):10s}  n = {row['count']:5d}   baseline = {b:.4f}")


N = 28884, classes = 3, n_min = 877, n_max = 26658
Imbalance ratio: 30.4x  (⚠ flag — run robustness sweep in Step 5)
Uniform random baseline: 1/3 = 0.333

Per-class random baseline  (n_class / (N−1)):
  NON-BINDING  n = 26658   baseline = 0.9230
  REGULAR-BINDING  n =  1349   baseline = 0.0467
  CRYPTIC-BINDING  n =   877   baseline = 0.0304


---
## Step 3 — Set the k range

Derive a justified k range from Steps 1–2. Hard upper bound: **k < n_min**. Soft lower bound:
**k ≥ 30** (or n_min−1 if smaller). If hubness is high, apply a more conservative ceiling.


In [73]:
# ── Step 3: derive k range ────────────────────────────────────────────────────
N_total  = len(metadata)
n_min    = int(metadata[feature].value_counts().min())
rhi_max  = hub.data["rhi"]["RobinHoodIndex"].max()

k_hard   = n_min - 1                     # strict upper bound (k must be < n_min)
k_min    = min(5, k_hard)                # soft lower bound: 5, capped at hard upper bound
k_sqrt   = int(np.sqrt(N_total))         # soft upper bound for large N
k_pct    = max(1, int(0.05 * N_total))   # 5% of N soft upper bound

if k_hard < k_min:
    print(f"⚠  n_min = {n_min}: hard upper bound (k < {n_min}) is below the lower bound "
          f"({k_min}). Clamping k range to [{k_min}] — interpret with caution.")

if rhi_max > 0.3:
    k_ceiling = max(k_min, min(k_hard, k_pct // 2))
    print(f"⚠  High hubness (max RHI = {rhi_max:.3f}): applying conservative k ceiling")
else:
    k_ceiling = max(k_min, min(k_hard, max(k_sqrt, k_pct)))

if k_ceiling < 20:
    # Edge case: override to full dense grid regardless of k_min/k_ceiling
    k_values = [3, 5, 10, 15, 20]
else:
    k_values = [k for k in [1, 5, 10, 20, 30, 50, 100] if k_min <= k <= k_ceiling]
if not k_values:
    k_values = [k_min]

print(f"N = {N_total},  n_min = {n_min}")
print(f"k lower bound (min 5, capped):   {k_min}")
print(f"k hard upper bound (< n_min):    {k_hard}")
print(f"k soft upper bound (√N):          {k_sqrt}")
print(f"k ceiling applied:               {k_ceiling}")
print(f"k values to sweep:               {k_values}")
print(f"Distance metrics:                {distance_metrics}")

⚠  High hubness (max RHI = 0.337): applying conservative k ceiling
N = 28884,  n_min = 877
k lower bound (min 5, capped):   5
k hard upper bound (< n_min):    876
k soft upper bound (√N):          169
k ceiling applied:               722
k values to sweep:               [5, 10, 20, 30, 50, 100]
Distance metrics:                ['euclidean', 'cosine']


In [74]:
# ── Review / override k_values ────────────────────────────────────────────────
# k_values was derived automatically in the cell above.
# Edit the line below to override (e.g. if you want a finer or coarser sweep).
# k_values = [10, 20, 30, 50]

print(f"k_values: {k_values}")


k_values: [5, 10, 20, 30, 50, 100]


---
## Step 4 — Main analysis: KNN alignment across k and distance metrics

For each sequence, what fraction of its k nearest neighbours share the same enzyme class?
Plotted across the full k range (from Step 3) and all three distance metrics.

**Ranking stability:** consistent rankings across k and metrics → high confidence.
Crossings or inversions → parameter-sensitive; report results across the full range.


In [75]:
%reload_ext autoreload
%autoreload 2
from emmaemb.vizualisation import (
    plot_knn_alignment_across_k,
    plot_knn_alignment_vs_class_balance,
    plot_knn_alignment_vs_feature_noise,
)
fig, df_mean = plot_knn_alignment_across_k(
    emma=ema,
    feature=feature,
    k_values=k_values,
    metrics=distance_metrics,
    color_discrete_map=colors,
    show_random_baselines=True,
    elbow_detection=True,
    return_data=True,
)
fig.show()


Elbow k (max-distance-from-chord method):
Embedding             Metric  Elbow k  Score at elbow
     ESM2    Cosine distance       30           0.866
     ANKH    Cosine distance       30           0.874
  ProstT5    Cosine distance       30           0.885
   ProtT5    Cosine distance       30           0.877
     ESM2 Euclidean distance       30           0.879
     ANKH Euclidean distance       30           0.865
  ProstT5 Euclidean distance       30           0.868
   ProtT5 Euclidean distance       30           0.864


---
## Step 5 — Robustness to class imbalance and label noise

The Pla2g2 dataset has notable imbalance across enzyme classes — run both sweeps.

### 5a — Progressive rebalancing

Progressively downsample the majority class toward n_min and repeat KNN alignment.
Stable rankings → imbalance is not driving the result.
Changing rankings → the score was frequency-driven.


In [76]:
fig_balance = plot_knn_alignment_vs_class_balance(
    emma=ema,
    feature=feature,
    emb_spaces=models,
    k_values=k_values,
    metrics=["cosine"],
    n_balance_steps=8,
    seed=42,
    color_discrete_map=colors,
    show_random_baselines=True,
)
fig_balance.update_layout(width=700)
fig_balance.show()


### 5b — Label noise robustness

Progressively corrupt a fraction of labels while embedding geometry stays fixed.
This reveals how geometrically tight the class structure really is.


In [77]:
%reload_ext autoreload
%autoreload 2
from emmaemb.vizualisation import (
    plot_knn_alignment_across_k,
    plot_knn_alignment_vs_class_balance,
    plot_knn_alignment_vs_feature_noise,
)
fig_noise = plot_knn_alignment_vs_feature_noise(
    emma=ema,
    feature=feature,
    emb_spaces=models,
    k_values=k_values,
    metrics=["cosine"],
    n_noise_steps=10,
    n_repeats=5,
    seed=42,
    color_discrete_map=colors,
    show_random_baselines=True,
)
fig_noise.update_layout(width=700)
fig_noise.show()
